# 05 — External-corpus evaluation, Bootstrap diagnostics and promotion gate

Evaluate the same unified checkpoint on independent external validation/test splits for capability benchmarks. Evaluate Bootstrap fixtures separately only to diagnose data and modality pipelines. Bootstrap is excluded from tokenizer training, optimizer updates, and external generalization metrics; its reports never contribute to the promotion score. Both evaluation tracks preserve the same checkpoint and tokenizer hashes.


In [ ]:
import sys, json
from pathlib import Path

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "phone-research.yaml").is_file() and (candidate / "src" / "onecore").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate OneCore project root from {current}")

root = find_project_root()
print("OneCore project root:", root)
sys.path.insert(0, str(root / "src"))

from onecore.checkpoint import write_json_atomic
from onecore.evaluate import evaluate_checkpoint, promotion_gate
from onecore.promote import promote_candidate

config = root / "configs" / "phone-research.yaml"
checkpoint = root / "run-state" / "checkpoint.pt"
tokenizer = root / "run-state" / "tokenizer.json"
thresholds = root / "configs" / "promotion-thresholds.json"

external_classes = {"external_staged", "external_reviewed"}
bootstrap_classes = {"bootstrap", "synthetic_unverified", "synthetic_verified"}

validation_metrics = evaluate_checkpoint(
    root, config, checkpoint, tokenizer, thresholds,
    root / "run-state" / "evaluation.validation.json", split="validation",
    data_classes=external_classes,
)
test_metrics = evaluate_checkpoint(
    root, config, checkpoint, tokenizer, thresholds,
    root / "run-state" / "evaluation.test.json", split="test",
    data_classes=external_classes,
)

bootstrap_validation_metrics = evaluate_checkpoint(
    root, config, checkpoint, tokenizer, thresholds,
    root / "run-state" / "evaluation.bootstrap.validation.json", split="validation",
    data_classes=bootstrap_classes,
)
bootstrap_test_metrics = evaluate_checkpoint(
    root, config, checkpoint, tokenizer, thresholds,
    root / "run-state" / "evaluation.bootstrap.test.json", split="test",
    data_classes=bootstrap_classes,
)

metrics = dict(validation_metrics)
metrics["test_metrics"] = test_metrics
metrics["test_manifest_sha256"] = test_metrics.get("split_manifest_sha256")
metrics["test_split_independent"] = (
    validation_metrics.get("split_manifest_sha256") != test_metrics.get("split_manifest_sha256")
)
metrics["evaluation_bundle_id"] = (
    f"{validation_metrics['evaluation_id']}+{test_metrics['evaluation_id']}"
)
metrics["bootstrap_diagnostics"] = {
    "validation": bootstrap_validation_metrics,
    "test": bootstrap_test_metrics,
    "purpose": "deterministic curriculum and modality-pipeline diagnostics; not part of external generalization score",
}
write_json_atomic(root / "run-state" / "evaluation.json", metrics)

passed, failures = promotion_gate(metrics)
print(json.dumps({
    "external_validation": validation_metrics,
    "external_test": test_metrics,
    "bootstrap_validation": bootstrap_validation_metrics,
    "bootstrap_test": bootstrap_test_metrics,
    "promotion_gate_passed": passed,
    "failures": failures,
}, ensure_ascii=False, indent=2))
if passed:
    print(promote_candidate(root / "run-state" / "evaluation.json", root / "run-state" / "model-registry.json"))
else:
    print("PROMOTION BLOCKED. No candidate was merged or marked as released.")
